In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

#### Data Preprocessing

In [2]:
'''
link to dataset: https://www.kaggle.com/datasets/datascientist97/astronomical-data
'''
df = pd.read_csv('/Users/cela/Documents/Code/CS375/project/cleaned_star_data.csv')
df = df.dropna()

#encode categorical features ('Star Color' and 'Spectral Class') into numerical values using LabelEncoder
label_encoder = LabelEncoder()
df['Star Color'] = label_encoder.fit_transform(df['Star color'])
df['Spectral Class'] = label_encoder.fit_transform(df['Spectral Class'])

df = df.drop("Star color", axis=1)
df = df.replace(' ', pd.NA)
df["Temperature (K)"] = pd.to_numeric(df["Temperature (K)"], errors="coerce")
df["Luminosity(L/Lo)"] = pd.to_numeric(df["Luminosity(L/Lo)"], errors="coerce")
df["Radius(R/Ro)"] = pd.to_numeric(df["Radius(R/Ro)"], errors="coerce")
df["Absolute magnitude(Mv)"] = pd.to_numeric(df["Absolute magnitude(Mv)"], errors="coerce")
for column in df.columns:
    if df[column].dtype in ['float64', 'int64']:
        df[column].fillna(df[column].median(), inplace=True)

print(df.dtypes)
print(df.isnull().sum())
df.head()

Temperature (K)           float64
Luminosity(L/Lo)          float64
Radius(R/Ro)              float64
Absolute magnitude(Mv)    float64
Star type                 float64
Spectral Class              int64
Star Color                  int64
dtype: object
Temperature (K)           0
Luminosity(L/Lo)          0
Radius(R/Ro)              0
Absolute magnitude(Mv)    0
Star type                 0
Spectral Class            0
Star Color                0
dtype: int64


/var/folders/y_/g2pbtc8d1kgbb25kqx01_fsr0000gn/T/ipykernel_10448/3208162965.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[column].fillna(df[column].median(), inplace=True)


,Temperature (K),Luminosity(L/Lo),Radius(R/Ro),Absolute magnitude(Mv),Star type,Spectral Class,Star Color
1,3042.0,0.000500,0.1542,16.60,0.0,5,3
2,2600.0,0.000300,0.1020,18.70,0.0,5,3
3,2800.0,0.000200,0.7625,16.65,0.0,5,3
4,1939.0,0.000138,0.1030,20.06,0.0,5,3
5,2840.0,0.085000,0.1100,16.98,0.0,5,3


#### Seperating Features and Target

In [3]:
X = df.drop("Star type", axis=1) #Features: physical properties of stars
y = df["Star type"] #Target: Star classification: (0-Brown Dwarf, 1-Red Dwarf, 2-White Dwarf, 3-Main Sequence, 4-Supergiants, 5-Hypergiants)

#### Splitting Data

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,      #20% of data goes to testing
    random_state=42,   
)

#### Training the Random Forest Model

In [5]:
classifier = RandomForestClassifier(
    n_estimators = 100,
    criterion = 'entropy', 
    random_state = 42
)
classifier.fit(X_train, y_train)

RandomForestClassifier(criterion='entropy', random_state=42)

#### Predictions

In [6]:
y_pred = classifier.predict(X_test)

#### Evaluation

In [7]:
from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00         9
         1.0       1.00      1.00      1.00        10
         2.0       1.00      1.00      1.00         9
         3.0       1.00      1.00      1.00         5
         4.0       0.83      1.00      0.91         5
         5.0       1.00      0.90      0.95        10

    accuracy                           0.98        48
   macro avg       0.97      0.98      0.98        48
weighted avg       0.98      0.98      0.98        48

Confusion Matrix:
[[ 9  0  0  0  0  0]
 [ 0 10  0  0  0  0]
 [ 0  0  9  0  0  0]
 [ 0  0  0  5  0  0]
 [ 0  0  0  0  5  0]
 [ 0  0  0  0  1  9]]


#### Sample Prediction

In [8]:
import numpy as np
new_star = np.array([[5800, 1.0, 1.0, 4.8, 3, 4]])
prediction = classifier.predict(new_star)
print("Predicted Star Type:", prediction[0]) #output: Main-Sequence

Predicted Star Type: 3.0


/opt/miniconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(



Based on the results, the model is correct 98% of the time when predicting star types.
The F-1 score shows that across all classes, my model is consistently achieving near-perfect precision and recall. It’s not just skewed toward the most common class.

For class 4, the model always catches actual class 4 stars when they appear (recall = 1.00), but whenever it predicts “class 4”, it’s only 83% correct. This can also be seen in the confusion matrix where one star in class 5 was predicted as class 4.

Compared to KNN, it has behaved identically. This could because my dataset has distinct feature ranges for each star type (temperature, luminosity, radius, etc.). Both Random Forest and KNN can easily learn to separate classes when the boundaries are clear.